# Visualizador Gradio dos Parquets

Caderno operacional para abrir, no Google Colab, um app Gradio read-only de navegacao dos Parquets `textos_parlamentares/v1` no Google Drive. A mesma estrutura tambem funciona localmente contra os Parquets de samples.

O app permite filtrar registros, consultar tabela compacta sem `texto`, carregar texto integral por `texto_id` e gerar um grafico Altair anual com eixo duplo.

## 1. Montar Google Drive

No Colab, esta deve ser a primeira celula executavel. Fora do Colab, o caderno assume modo local e usa samples quando disponiveis.

In [ ]:
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
    print('Ambiente local detectado; Google Drive nao sera montado.')
else:
    IN_COLAB = True
    drive.mount('/content/drive')

## 2. Preparar repositorio

No Colab, o repositorio e clonado ou atualizado antes de importar codigo local. Defina `REPO_REF` apenas se quiser fixar branch, tag ou commit.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/pedblan/falando_nela.git'
REPO_DIR = Path('/content/falando_nela') if IN_COLAB else Path.cwd().resolve()
REPO_REF = ''  # opcional: branch, tag ou commit

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
    if REPO_REF:
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)
else:
    for candidate in [REPO_DIR, *REPO_DIR.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'notebooks').exists():
            REPO_DIR = candidate
            break

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repositorio:', REPO_DIR)

## 3. Instalar dependencias

Instala apenas dependencias de leitura, consulta, visualizacao e app. Esta celula nao executa coleta, normalizacao, conversao de arquivos nem escrita no Drive.

In [ ]:
import subprocess
import sys

PACKAGES = [
    'altair>=5,<6',
    'duckdb>=1,<2',
    'gradio>=4,<6',
    'pandas>=2,<3',
    'pyarrow>=15,<23',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], check=True)

## 4. Definir caminhos

No Colab, a entrada oficial e o diretorio de Parquets processados no Drive. Fora do Colab, o caderno usa os Parquets de samples locais.

In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path('/content/drive/MyDrive/falando_nela/data')
COLAB_PARQUET_ROOT = Path('/content/drive/MyDrive/falando_nela/data/processed/textos_parlamentares/v1/parquet')
SAMPLES_PARQUET_ROOT = REPO_DIR / 'data' / 'samples' / 'textos_parlamentares' / 'v1' / 'parquet'

DATA_PROFILE = 'colab-drive' if IN_COLAB else 'samples-local'
PARQUET_ROOT = COLAB_PARQUET_ROOT if IN_COLAB else SAMPLES_PARQUET_ROOT
os.environ['FALANDO_NELA_DATA_ROOT'] = str(DATA_ROOT)

print('Perfil:', DATA_PROFILE)
print('PARQUET_ROOT:', PARQUET_ROOT)

## 5. Importar visualizador

A logica do app fica em `processamento.visualizador_parquets`, para que a consulta possa ser testada localmente e usada do mesmo modo no Colab.

In [ ]:
import importlib

import gradio as gr
import altair as alt
import duckdb

import processamento.visualizador_parquets as visualizador_parquets
visualizador_parquets = importlib.reload(visualizador_parquets)

from processamento.visualizador_parquets import (
    build_gradio_app,
    build_yearly_metrics_chart,
    fetch_text_by_id,
    list_parquet_files,
    query_compact_table,
    query_yearly_metrics,
)

print('gradio:', gr.__version__)
print('altair:', alt.__version__)
print('duckdb:', duckdb.__version__)

## 6. Conferir Parquets disponiveis

A lista e descoberta dinamicamente. Bases novas devem aparecer aqui depois que os Parquets forem gerados no fluxo apropriado e o caderno for relancado ou a lista do app atualizada.

In [ ]:
parquet_files = list_parquet_files(PARQUET_ROOT)
print(f'{len(parquet_files)} Parquet(s) encontrado(s) em {PARQUET_ROOT}')
for name in parquet_files:
    print('-', name)

if not parquet_files:
    raise FileNotFoundError(
        'Nenhum Parquet encontrado. No Colab, confirme se o Drive esta montado e se os arquivos existem em '
        '/content/drive/MyDrive/falando_nela/data/processed/textos_parlamentares/v1/parquet/'
    )

## 7. Smoke read-only da consulta

Este smoke consulta a primeira base encontrada, respeitando o limite de linhas da tabela compacta e gerando o grafico anual. Ele nao altera nenhum arquivo.

In [ ]:
SMOKE_PARQUET = parquet_files[0]
SMOKE_LIMIT = 5
SMOKE_BUSCA = ''  # exemplos: 'plexo', '"saude publica"', 'plexo -complexo'

compact_df, query_info = query_compact_table(
    PARQUET_ROOT,
    SMOKE_PARQUET,
    busca_textual=SMOKE_BUSCA,
    limit=SMOKE_LIMIT,
    sort_column='data',
)
print(query_info.status)
print('Coluna texto na tabela compacta?', 'texto' in compact_df.columns)
display(compact_df)

metrics_df = query_yearly_metrics(PARQUET_ROOT, SMOKE_PARQUET, busca_textual=SMOKE_BUSCA)
chart = build_yearly_metrics_chart(metrics_df)
display(chart)

## 8. Smoke de texto integral

Seleciona o primeiro `texto_id` retornado no smoke anterior e carrega o texto integral sob demanda.

In [ ]:
if compact_df.empty or 'texto_id' not in compact_df.columns:
    print('Sem texto_id disponivel no smoke da tabela compacta.')
else:
    smoke_texto_id = str(compact_df.iloc[0]['texto_id'])
    lookup = fetch_text_by_id(PARQUET_ROOT, SMOKE_PARQUET, smoke_texto_id)
    print(lookup.status)
    print('Metadados:', lookup.metadata)
    print('Tamanho do texto:', len(lookup.text))
    print(lookup.text[:1000])

## 9. Iniciar app Gradio

O app e read-only. Ele le os Parquets selecionados, aplica filtros com DuckDB, mostra a tabela compacta sem `texto`, carrega texto integral por `texto_id` e exibe o grafico Altair anual com eixo Y duplo.

In [ ]:
app = build_gradio_app(PARQUET_ROOT)
if IN_COLAB:
    app.launch(share=True)
else:
    app.launch(share=False)